### Reading cleaned csv files

In [9]:
import pandas as pd
import sqlite3
import re

# 1. Read your cleaned CSV files
prices_raw = pd.read_csv("cleaned_prices.csv")
prices_adjusted = pd.read_csv("cleaned_prices_split_adjusted.csv")
securities = pd.read_csv("cleaned_securities.csv")
fundamentals = pd.read_csv("cleaned_fundamentals.csv")

### Remove the exported row-index column

In [10]:
fundamentals = fundamentals.drop(
    columns=["Unnamed: 0"], errors="ignore"
)

### Standardize column names

In [11]:
def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(
            r"_+", "_",
            re.sub(r"[^a-z0-9]+", "_", col.strip().lower())
        ).strip("_")
        for col in df.columns
    ]
    return df


prices_raw = standardize_columns(prices_raw)
prices_adjusted = standardize_columns(prices_adjusted)
securities = standardize_columns(securities)
fundamentals = standardize_columns(fundamentals)

### Use the same column name for the company ticker

In [12]:
securities = securities.rename(columns={"ticker_symbol": "symbol"})
fundamentals = fundamentals.rename(columns={"ticker_symbol": "symbol"})

### Standardize date formats

In [13]:
prices_raw["date"] = pd.to_datetime(
    prices_raw["date"]
).dt.strftime("%Y-%m-%d")

prices_adjusted["date"] = pd.to_datetime(
    prices_adjusted["date"]
).dt.strftime("%Y-%m-%d")

fundamentals["period_ending"] = pd.to_datetime(
    fundamentals["period_ending"]
).dt.strftime("%Y-%m-%d")

securities["date_first_added"] = pd.to_datetime(
    securities["date_first_added"], errors="coerce"
).dt.strftime("%Y-%m-%d")

###  Dinfining database tables

In [14]:
tables = {
    "securities": securities,
    "prices_unadjusted": prices_raw,
    "prices_adjusted": prices_adjusted,
    "fundamentals": fundamentals
}

primary_keys = {
    "securities": ["symbol"],
    "prices_unadjusted": ["date", "symbol"],
    "prices_adjusted": ["date", "symbol"],
    "fundamentals": ["symbol", "period_ending"]
}

### Create the SQLite database and tables

In [ ]:
with sqlite3.connect("stocks.db") as conn:
    conn.execute("PRAGMA foreign_keys = ON")

    for table_name, df in tables.items():
        columns = []

        for column in df.columns:
            dtype = df[column].dtype

            if pd.api.types.is_integer_dtype(dtype):
                sql_type = "INTEGER"
            elif pd.api.types.is_numeric_dtype(dtype):
                sql_type = "REAL"
            else:
                sql_type = "TEXT"

            not_null = (
                " NOT NULL"
                if column in primary_keys[table_name]
                else ""
            )

            columns.append(
                f'"{column}" {sql_type}{not_null}'
            )

        # Defining primary keys
        key_columns = ", ".join(
            f'"column"' .replace("column", key)
            for key in primary_keys[table_name]
        )

        columns.append(f"PRIMARY KEY ({key_columns})")

        # Link each data table to the company table
        if table_name != "securities":
            columns.append(
                'FOREIGN KEY ("symbol") '
                'REFERENCES "securities" ("symbol")'
            )

        schema = ", ".join(columns)

        conn.execute(
            f'CREATE TABLE "{table_name}" ({schema})'
        )

    # 6. Loading the data into  tables
    for table_name, df in tables.items():
        df.to_sql(
            table_name,
            conn,
            if_exists="append",
            index=False,
            chunksize=5000
        )

        print(f"Loaded {len(df):,} rows into {table_name}")

print("\nDatabase created successfully: stocks.db")

Loaded 448 rows into securities
Loaded 760,913 rows into prices_unadjusted
Loaded 760,913 rows into prices_adjusted
Loaded 1,719 rows into fundamentals

Database created successfully: stocks.db
